# Optimización de Pagos — FV Procesados
### Notebook de desarrollo e iteración (Modelos 2, 3 y 5)

Este notebook reutiliza los datos reales del dashboard (`Datos flujo de efectivo.xlsx` y
`Pagos pendientes.xlsx`) para desarrollar, probar y comparar los modelos de optimización
de pagos **fuera** de Streamlit, antes de integrarlos de vuelta al dashboard.

**Estado actual de este notebook:**

| Modelo | Estado |
|---|---|
| Modelo 5 — Mínimo ratio deuda/caja | ✅ Implementado (Opción A: heurística mejorada, Opción B: SLP) |
| Modelo 2 — Estabilidad de caja | 🟡 Pendiente (solo estructura en markdown) |
| Modelo 3 — Máxima resiliencia | 🟡 Pendiente (solo estructura en markdown) |

> Este notebook será iterado por otra IA. Al regresar, debe verificarse que:
> 1. Las funciones de carga siguen coincidiendo con el formato de los archivos Excel del dashboard.
> 2. Los resultados de las Opciones A y B del Modelo 5 sean coherentes (ratio D/C decreciente, caja nunca negativa).
> 3. No se haya modificado la firma de las funciones de carga (`cargar_datos_flujo`, `cargar_pagos_pendientes`, etc.) sin actualizar también `app.py`.


In [1]:
# ============================================================
#  IMPORTS
# ============================================================
import numpy as np
import pandas as pd
from scipy.optimize import linprog, minimize
import matplotlib.pyplot as plt

pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.width", None)

In [2]:
# ============================================================
#  CONFIGURACIÓN — MISMOS ARCHIVOS QUE USA EL DASHBOARD
# ============================================================
# IMPORTANTE: estas rutas deben apuntar a los mismos archivos que usa
# app.py (Streamlit). Si el dashboard corre en otra carpeta, ajustar aquí.
ARCHIVO_DATOS = "../Datos flujo de efectivo.xlsx"
ARCHIVO_PAGOS = "../Pagos pendientes.xlsx"

COLUMNAS_PAGOS = [
    "concepto", "categoría", "tipo", "importe", "semana",
    "inicio_semana", "fecha_registro", "estatus", "fecha_pago",
]


## 1. Carga de datos

Estas funciones son una copia fiel de la lógica usada en `app.py`, para garantizar que
lo que probemos aquí sea 100% consistente con lo que el dashboard va a ejecutar cuando
integremos el código de vuelta.


In [3]:
def cargar_datos_flujo(ruta=ARCHIVO_DATOS):
    """Carga el histórico de movimientos (mismo formato que usa generar_flujo en el dashboard)."""
    df = pd.read_excel(ruta)
    df["inicio_semana"] = pd.to_datetime(df["inicio_semana"], dayfirst=True)
    df["tipo"] = df["tipo"].astype(str).str.strip()
    df["categoría"] = df["categoría"].astype(str).str.strip()
    df["descripción"] = df["descripción"].astype(str).str.strip()
    df["importe"] = pd.to_numeric(df["importe"], errors="coerce").fillna(0)
    return df


def cargar_pagos_pendientes(ruta=ARCHIVO_PAGOS):
    """Carga pagos pendientes. Si el archivo no existe, retorna un DataFrame vacío
    con las columnas esperadas (igual que cargar_pagos() en el dashboard)."""
    try:
        dfp = pd.read_excel(ruta)
        for col in COLUMNAS_PAGOS:
            if col not in dfp.columns:
                dfp[col] = ""
        dfp = dfp[COLUMNAS_PAGOS]
    except FileNotFoundError:
        dfp = pd.DataFrame(columns=COLUMNAS_PAGOS)
    return dfp


def obtener_deudas_pendientes(dfp):
    """Agrupa la deuda pendiente por concepto/proveedor (D_i). Igual que en el dashboard."""
    pend = dfp[dfp["estatus"] == "Pendiente"].copy()
    pend["importe"] = pd.to_numeric(pend["importe"], errors="coerce").fillna(0)
    resumen = pend.groupby("concepto")["importe"].sum()
    return resumen[resumen > 0].sort_values(ascending=False)


def calcular_saldo_actual(df):
    ingresos = df.loc[df["tipo"] == "Ingreso", "importe"].sum()
    egresos = df.loc[df["tipo"] == "Egreso", "importe"].sum()
    return float(ingresos - egresos)


def obtener_proyeccion_base(df, horizonte):
    """Proyecta ingresos/egresos futuros usando el promedio de las últimas 6 semanas
    registradas. Reescrito con pivot_table para evitar el FutureWarning de
    groupby().apply() que tiene la versión del dashboard."""
    resumen = df.pivot_table(
        index="inicio_semana", columns="tipo", values="importe",
        aggfunc="sum", fill_value=0,
    ).sort_index()

    for col in ["Ingreso", "Egreso"]:
        if col not in resumen.columns:
            resumen[col] = 0.0

    ultimas = resumen.tail(6)
    ingreso_prom = float(ultimas["Ingreso"].mean()) if not ultimas.empty else 0.0
    egreso_prom = float(ultimas["Egreso"].mean()) if not ultimas.empty else 0.0
    return [ingreso_prom] * horizonte, [egreso_prom] * horizonte


In [4]:
# ============================================================
#  CARGA REAL DE DATOS
# ============================================================
df = cargar_datos_flujo()
dfp = cargar_pagos_pendientes()

deudas = obtener_deudas_pendientes(dfp)
saldo_actual = calcular_saldo_actual(df)

print(f"Movimientos históricos cargados: {len(df)}")
print(f"Pagos pendientes registrados: {len(dfp)}")
print(f"Deudas pendientes por concepto (D_i):")
display(deudas)
print(f"\nSaldo actual (C0): {saldo_actual:,.2f}")


Movimientos históricos cargados: 540
Pagos pendientes registrados: 32
Deudas pendientes por concepto (D_i):


concepto
CAJA RUTH GDL                                   77,738.85
IMSS CXM                                        73,046.60
SINDULFO                                        49,247.80
ROBERTO REYES                                   49,247.80
RAISA                                           40,000.00
AVV                                             39,000.00
Ing Aguilar , embolsadora                       35,178.00
CIELO (TARIMAS)                                 32,480.00
Cristian Ramirez                                28,500.00
IMSS                                            25,000.00
EFECTIVO                                        20,000.00
NOMINA JACONA Y OCUMICHO SABADO                 20,000.00
TARIMA                                          16,240.00
JOSE HDEZ GARCIA (PAGO TARIMA)                  15,370.00
GLADYS (CERTIFICACION)                          15,000.00
LORE (Transporte Personal)                      12,000.00
GASTOS                                          10,000.00
DEPAC


Saldo actual (C0): -137,269.68


In [5]:
# ============================================================
#  PARÁMETROS DE PRUEBA (ajustar libremente en cada corrida)
# ============================================================
HORIZONTE = 8          # T, semanas de planeación
L_MIN = round(max(saldo_actual * 0.1, 0.0), 2)   # caja mínima deseada

I, E = obtener_proyeccion_base(df, HORIZONTE)
C0 = saldo_actual
conceptos = deudas.index.tolist()
D0 = deudas.values.tolist()

print(f"Horizonte: {HORIZONTE} semanas")
print(f"L_min: {L_MIN:,.2f}")
print(f"Ingreso proyectado/semana: {I[0]:,.2f}")
print(f"Egreso proyectado/semana: {E[0]:,.2f}")
print(f"C0: {C0:,.2f}")
print(f"Conceptos con deuda: {conceptos}")
print(f"D0: {[round(d,2) for d in D0]}")


Horizonte: 8 semanas
L_min: 0.00
Ingreso proyectado/semana: 233,325.97
Egreso proyectado/semana: 253,671.07
C0: -137,269.68
Conceptos con deuda: ['CAJA RUTH GDL', 'IMSS CXM', 'SINDULFO', 'ROBERTO REYES', 'RAISA', 'AVV', 'Ing Aguilar , embolsadora', 'CIELO (TARIMAS)', 'Cristian Ramirez', 'IMSS', 'EFECTIVO', 'NOMINA JACONA Y OCUMICHO SABADO', 'TARIMA', 'JOSE HDEZ GARCIA (PAGO TARIMA)', 'GLADYS (CERTIFICACION)', 'LORE (Transporte Personal)', 'GASTOS', 'DEPACHO GARNICA', 'Edgar Joel Vega Martínez (Emplaye)', 'Bolsa Transparente (Chavinda)', 'IMPUESTOS LAR', 'LORE', 'CIRAM', 'PLATANO 1A/2A RECEPCION (12/11/25)/(14/11/25)', 'CIPSA', 'SIMA (Calibracion)', 'DIKEN', 'LAR seguro aveo', 'Contador Reymundo', 'BOLSA', 'telefono']
D0: [77738.85, 73046.6, 49247.8, 49247.8, 40000.0, 39000.0, 35178.0, 32480.0, 28500.0, 25000.0, 20000.0, 20000.0, 16240.0, 15370.0, 15000.0, 12000.0, 10000.0, 8952.3, 8129.69, 8000.0, 7900.0, 6970.0, 6050.0, 5253.25, 3886.0, 2500.0, 2033.0, 2028.84, 1000.0, 593.5, 246.44]


---
## 2. Modelo 5 — Mínimo ratio deuda/caja

**Objetivo declarado:** `min Σ_t D_t / C_t`

Este es el modelo que discutimos a fondo: el ratio `D_t/C_t` es **no convexo**, así que
implementamos las dos rutas que hablamos:

- **Opción A — Heurística mejorada:** rápida, explicable, ahora sí calcula el ratio real
  en cada paso y prioriza liquidar deudas pequeñas por completo en vez de repartir
  proporcionalmente. Sirve como baseline y como fallback si la Opción B no converge.
- **Opción B — SLP (Sequential Linear Programming):** aproximación real al problema de
  minimizar el ratio, linealizando alrededor de la solución actual y resolviendo un LP en
  cada iteración, unas pocas veces hasta converger.

Ambas opciones devuelven la misma estructura (`P`, matriz de pagos `n x T`) para que se
puedan comparar directamente con `construir_tabla_recomendacion`.

**Correcciones aplicadas respecto a la versión del dashboard:**
1. El ratio `D_t/C_t` ahora se calcula explícitamente y se expone en la tabla de salida.
2. Se usa `D_t/(C_t + L_min)` en vez de `D_t/C_t` puro, para evitar división por valores
   cercanos a cero cuando la caja está muy ajustada.
3. Opción A prioriza liquidar deudas pequeñas completas primero (reduce el número de
   acreedores activos más rápido), en vez de repartir proporcionalmente al tamaño.
4. Guardia explícita de caja negativa: si `C_disponible < L_min`, el pago efectivo de esa
   semana es 0, de forma auditable (no solo por el `max(0, ...)` implícito).


In [6]:
# ============================================================
#  MODELO 5 — OPCIÓN A: Heurística mejorada
# ============================================================
def modelo5_opcion_a(D0, I, E, C0, L_min, T, agresividad=0.5, epsilon=1e-6):
    """
    Heurística secuencial mejorada para minimizar el ratio D_t/(C_t+L_min).

    Diferencias clave vs. la versión original del dashboard:
    - Prioriza liquidar POR COMPLETO las deudas más pequeñas primero (en vez de
      repartir proporcionalmente al tamaño de cada deuda).
    - Calcula y devuelve el ratio real D_t/(C_t+L_min) semana a semana.
    - Guardia explícita: si no hay excedente sobre L_min, no se paga esa semana.

    Retorna:
        P: matriz (n, T) de pagos por concepto y semana
        ratios: lista de longitud T con el ratio D_t/(C_t+L_min) tras el pago de esa semana
        exito: bool, siempre True (es una heurística, no un solver)
    """
    n = len(D0)
    IE = np.array(I, dtype=float) - np.array(E, dtype=float)
    D_restante = np.array(D0, dtype=float)
    orden_prioridad = np.argsort(D_restante)  # deudas más pequeñas primero

    C = C0
    P = np.zeros((n, T))
    ratios = []

    for t in range(T):
        C_disponible = C + IE[t]
        excedente = max(0.0, C_disponible - L_min)

        pago_total = min(excedente * agresividad, D_restante.sum())

        if pago_total > 0 and D_restante.sum() > 0:
            restante_a_pagar = pago_total
            for i in orden_prioridad:
                if restante_a_pagar <= 0:
                    break
                abono = min(D_restante[i], restante_a_pagar)
                P[i, t] = abono
                D_restante[i] -= abono
                restante_a_pagar -= abono

        C = C_disponible - P[:, t].sum()
        ratio_t = D_restante.sum() / (C + L_min + epsilon)
        ratios.append(ratio_t)

    return P, ratios, True


In [8]:
# ============================================================
#  TABLA DE RESULTADOS (incluye el ratio, a diferencia del dashboard actual)
# ============================================================
def construir_tabla_modelo5(P, ratios, conceptos, I, E, C0, L_min):
    n, T = P.shape
    IE = np.array(I, dtype=float) - np.array(E, dtype=float)
    pagos_sem = P.sum(axis=0)
    C = C0 + np.cumsum(IE) - np.cumsum(pagos_sem)
    U = IE - pagos_sem

    filas = []
    for t in range(T):
        fila = {"Semana": t + 1}
        for i, concepto in enumerate(conceptos):
            fila[f"Pagar {concepto} ($)"] = round(float(P[i, t]), 2)
        fila["Utilidad/Retenida ($)"] = round(float(U[t]), 2)
        fila["Caja Proyectada ($)"] = round(float(C[t]), 2)
        fila["Ratio D/C"] = round(float(ratios[t]), 4)
        if C[t] <= L_min:
            estado = "Sin margen"
        elif C[t] <= L_min * 1.5:
            estado = "Ajustado"
        else:
            estado = "Operable"
        fila["Estado"] = estado
        filas.append(fila)

    return pd.DataFrame(filas)


In [9]:
# ============================================================
#  PRUEBA — OPCIÓN A
# ============================================================
AGRESIVIDAD = 0.5

P_a, ratios_a, exito_a = modelo5_opcion_a(D0, I, E, C0, L_MIN, HORIZONTE, agresividad=AGRESIVIDAD)
tabla_a = construir_tabla_modelo5(P_a, ratios_a, conceptos, I, E, C0, L_MIN)

print("Modelo 5 - Opción A (heurística mejorada)")
display(tabla_a)


Modelo 5 - Opción A (heurística mejorada)


,Semana,Pagar CAJA RUTH GDL ($),Pagar IMSS CXM ($),Pagar SINDULFO ($),Pagar ROBERTO REYES ($),Pagar RAISA ($),Pagar AVV ($),"Pagar Ing Aguilar , embolsadora ($)",Pagar CIELO (TARIMAS) ($),Pagar Cristian Ramirez ($),Pagar IMSS ($),Pagar EFECTIVO ($),Pagar NOMINA JACONA Y OCUMICHO SABADO ($),Pagar TARIMA ($),Pagar JOSE HDEZ GARCIA (PAGO TARIMA) ($),Pagar GLADYS (CERTIFICACION) ($),Pagar LORE (Transporte Personal) ($),Pagar GASTOS ($),Pagar DEPACHO GARNICA ($),Pagar Edgar Joel Vega Martínez (Emplaye) ($),Pagar Bolsa Transparente (Chavinda) ($),Pagar IMPUESTOS LAR ($),Pagar LORE ($),Pagar CIRAM ($),Pagar PLATANO 1A/2A RECEPCION (12/11/25)/(14/11/25) ($),Pagar CIPSA ($),Pagar SIMA (Calibracion) ($),Pagar DIKEN ($),Pagar LAR seguro aveo ($),Pagar Contador Reymundo ($),Pagar BOLSA ($),Pagar telefono ($),Utilidad/Retenida ($),Caja Proyectada ($),Ratio D/C,Estado
0,1,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,"-20,345.10","-157,614.78",-3.94,Sin margen
1,2,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,"-20,345.10","-177,959.88",-3.49,Sin margen
2,3,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,"-20,345.10","-198,304.98",-3.13,Sin margen
3,4,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,"-20,345.10","-218,650.08",-2.84,Sin margen
4,5,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,"-20,345.10","-238,995.18",-2.60,Sin margen
5,6,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,"-20,345.10","-259,340.28",-2.40,Sin margen
6,7,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,"-20,345.10","-279,685.38",-2.22,Sin margen
7,8,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,"-20,345.10","-300,030.48",-2.07,Sin margen


### Checklist de verificación para la IA que retome este notebook

- [ ] ¿La caja proyectada nunca cae por debajo de `L_min` en la Opción B? (restricción dura)
- [ ] ¿La caja proyectada respeta `L_min` de forma razonable en la Opción A? (es solo un
      objetivo blando ahí, puede acercarse pero no debería quedar muy por debajo)
- [ ] ¿El ratio D/C es monótonamente no creciente o al menos tiende a bajar en ambas opciones?
- [ ] ¿La suma de pagos por concepto nunca excede `D0[i]` en ninguna de las dos opciones?
- [ ] ¿Con `agresividad` baja (ej. 0.1) la Opción A sigue siendo estable (no diverge, no paga de más)?
- [ ] Revisar si el gradiente aproximado de la Opción B (`costo_marginal`) necesita ajustarse
      si se cambia la definición del ratio (por ejemplo, si se decide usar `D_t/C_t` puro
      en vez de `D_t/(C_t+L_min)`).
- [ ] Considerar agregar una tercera opción: comparar contra "no pagar nada" como caso base,
      para cuantificar el valor agregado de cada estrategia.


---
## 3. Modelo 2 — Estabilidad de caja *(pendiente — solo estructura)*

> **No implementar código todavía.** Esta sección queda preparada para cuando retomemos
> este modelo en la conversación.

**Fórmula objetivo:** `min α·Σ_t (C_t − C̄)² + Σ_i D_i,final`

**Diagnóstico ya discutido (ver conversación previa) — pendiente de resolver aquí:**

1. Sin restricción dura `C_t >= L_min` — solo hay `C_t >= 0`. Falta amarrar el parámetro
   `L_min` que ya existe en la UI del dashboard a una restricción real del solver.
2. Punto inicial `x0 = np.zeros(n*T)` es débil para SLSQP con restricciones no triviales;
   se evaluará repartir la deuda uniformemente entre semanas como arranque, igual que se
   hizo en la Opción B del Modelo 5.
3. No hay control sobre el *ritmo* de pago — el término `β·Σ D_i,final` no penaliza pagos
   súbitos concentrados en una sola semana, lo cual puede ir en contra del objetivo de
   "estabilidad".
4. `res.success` como único criterio de validez — se evaluará correr múltiples arranques
   (`x0`) y quedarse con el mejor, o migrar a `trust-constr`.

**Pendiente de decidir en la próxima sesión:**
- ¿Se agrega restricción dura de `L_min` sí o sí, o se deja como término penalizado blando
  igual que ahora (pero corregido)?
- ¿Se necesita suavizar el ritmo de pago con un término adicional (ej. penalizar varianza
  de los pagos entre semanas, no solo de la caja)?
- ¿Qué solver usamos: mantener SLSQP con múltiples arranques, o migrar a `trust-constr`?


---
## 4. Modelo 3 — Máxima resiliencia financiera *(pendiente — solo estructura)*

> **No implementar código todavía.** Esta sección queda preparada para cuando retomemos
> este modelo en la conversación.

**Fórmula objetivo:** `max min_t C_t` (maximin de caja), vía formulación de epígrafe LP.

**Diagnóstico ya discutido (ver conversación previa) — pendiente de resolver aquí:**

1. Es matemáticamente el más "limpio" de los cinco (LP puro, solución global garantizada),
   pero **no tiene ningún incentivo a pagar deuda**. Si no pagar nada ya deja un buen piso
   de caja, el óptimo trivial es `P = 0` — inútil justo cuando la empresa está sana y
   "debería" ir abonando.
2. No hay preferencia entre deudas cuando sí conviene pagar algo — el LP puede repartir de
   cualquier forma entre las deudas empatadas en el óptimo, sin ningún criterio de
   priorización.

**Pendiente de decidir en la próxima sesión:**
- ¿Se agrega un término secundario en el objetivo (lexicográfico: primero maximizar el
  piso de caja, y como criterio de desempate, maximizar el pago total de deuda) para
  evitar el óptimo trivial de "no pagar nada"?
- ¿Qué criterio de prioridad usamos entre deudas cuando hay margen para pagar (tamaño,
  antigüedad, o el mismo orden "deudas pequeñas primero" que usamos en el Modelo 5 Opción A)?
- ¿Vale la pena exponer también el "piso de caja" (`m`) como una métrica visible en la
  tabla de resultados, similar al ratio D/C que ya agregamos al Modelo 5?
